In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

def upload(file_path):
    with open(file_path, "rb") as f:
        return client.beta.files.upload(file=f)

def download_file(file_id, output_path):
    data = client.beta.files.download(file_id)
    content = data.read()
    with open(output_path, "wb") as f:
        f.write(content)
    print(f"Downloaded: {output_path} ({len(content)} bytes)")

def chat(messages, tools=None):
    params = {"model": model, "max_tokens": 16000, "messages": messages}
    if tools: params["tools"] = tools
    return client.messages.create(**params)

In [ ]:
# Step 1: Upload CSV via Files API
file_metadata = upload("streaming.csv")
print(f"File ID: {file_metadata.id}")

In [ ]:
# Step 2: Analyze churn with code execution
messages = [{"role": "user", "content": [
    {"type": "text", "text": (
        "Run a detailed analysis to determine major drivers of churn. "
        "Your final output should include at least one detailed plot "
        "summarizing your findings."
    )},
    {"type": "container_upload", "file_id": file_metadata.id},
]}]

response = chat(
    messages,
    tools=[{"type": "code_execution_20250522", "name": "code_execution"}],
)

In [ ]:
# Step 3: Inspect response blocks
for i, block in enumerate(response.content):
    print(f"Block {i}: type={block.type}")
    if block.type == "text":
        print(f"  text: {block.text[:200]}...")
    elif block.type == "server_tool_use":
        print(f"  code: {block.input.get('code', '')[:150]}...")
    elif block.type == "code_execution_tool_result":
        result = block.content
        if result.stdout:
            print(f"  stdout: {result.stdout[:200]}")
        if result.stderr:
            print(f"  stderr: {result.stderr[:200]}")
        for item in result.content:
            if item.type == "code_execution_output":
                print(f"  generated file: {item.file_id}")
                download_file(item.file_id, "churn_analysis.png")

In [ ]:
# Step 4: Print final analysis text
for block in response.content:
    if block.type == "text" and len(block.text) > 100:
        print(block.text[:2000])
        break